# Introduction to Kubernetes Assignment

Covers all 4 sub-tasks:
1. Kubernetes setup (Minikube / managed service)
2. Application deployment
3. Resource management (Pods, Services, Deployments)
4. Helm charts

## 1. Kubernetes Setup
**Option A — Local (Minikube):**
```bash
curl -LO https://storage.googleapis.com/minikube/releases/latest/minikube-linux-amd64
sudo install minikube-linux-amd64 /usr/local/bin/minikube
minikube start --driver=docker
kubectl cluster-info
kubectl get nodes
```

**Option B — Managed service (GKE example):**
```bash
gcloud container clusters create demo-cluster --num-nodes=3 --zone=us-central1-a
gcloud container clusters get-credentials demo-cluster --zone=us-central1-a
kubectl get nodes
```
(EKS is analogous via `eksctl create cluster`.)

## 2. Application Deployment

In [1]:
import os
os.makedirs('project/k8s', exist_ok=True)

with open('project/k8s/deployment.yaml', 'w') as f:
    f.write('''apiVersion: apps/v1
kind: Deployment
metadata:
  name: demo-app
  labels:
    app: demo-app
spec:
  replicas: 3
  selector:
    matchLabels:
      app: demo-app
  template:
    metadata:
      labels:
        app: demo-app
    spec:
      containers:
        - name: demo-app
          image: nginxdemos/hello
          ports:
            - containerPort: 80
          resources:
            requests: { cpu: "100m", memory: "64Mi" }
            limits:   { cpu: "250m", memory: "128Mi" }
''')

with open('project/k8s/service.yaml', 'w') as f:
    f.write('''apiVersion: v1
kind: Service
metadata:
  name: demo-app-svc
spec:
  type: NodePort
  selector:
    app: demo-app
  ports:
    - port: 80
      targetPort: 80
      nodePort: 30080
''')
print("Created project/k8s/{deployment.yaml, service.yaml}")
print("Deploy:  kubectl apply -f project/k8s/deployment.yaml -f project/k8s/service.yaml")
print("Access:  minikube service demo-app-svc --url")


Created project/k8s/{deployment.yaml, service.yaml}
Deploy:  kubectl apply -f project/k8s/deployment.yaml -f project/k8s/service.yaml
Access:  minikube service demo-app-svc --url


## 3. Resource Management — Pods, Services, Deployments
```bash
kubectl get pods -o wide
kubectl describe pod <pod-name>
kubectl get deployments
kubectl get services

# Scale the deployment
kubectl scale deployment demo-app --replicas=5

# Rolling update to a new image
kubectl set image deployment/demo-app demo-app=nginxdemos/hello:latest
kubectl rollout status deployment/demo-app
kubectl rollout undo deployment/demo-app   # rollback if needed

# Logs / exec
kubectl logs <pod-name>
kubectl exec -it <pod-name> -- /bin/sh

# Cleanup
kubectl delete -f project/k8s/
```

## 4. Helm Charts
Package the same app as a Helm chart so it can be templated and versioned.

In [2]:
helm_cmds = '''
# Install Helm
curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

# Scaffold a new chart
helm create demo-app-chart
'''
print(helm_cmds)



# Install Helm
curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

# Scaffold a new chart
helm create demo-app-chart



In [3]:
import os
os.makedirs('project/demo-app-chart/templates', exist_ok=True)

with open('project/demo-app-chart/Chart.yaml', 'w') as f:
    f.write('''apiVersion: v2
name: demo-app-chart
description: A Helm chart for the demo app
version: 0.1.0
appVersion: "1.0"
''')

with open('project/demo-app-chart/values.yaml', 'w') as f:
    f.write('''replicaCount: 3
image:
  repository: nginxdemos/hello
  tag: latest
service:
  type: NodePort
  port: 80
  nodePort: 30080
''')

with open('project/demo-app-chart/templates/deployment.yaml', 'w') as f:
    f.write('''apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ .Release.Name }}-demo-app
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      app: {{ .Release.Name }}-demo-app
  template:
    metadata:
      labels:
        app: {{ .Release.Name }}-demo-app
    spec:
      containers:
        - name: demo-app
          image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"
          ports:
            - containerPort: 80
''')

with open('project/demo-app-chart/templates/service.yaml', 'w') as f:
    f.write('''apiVersion: v1
kind: Service
metadata:
  name: {{ .Release.Name }}-demo-app-svc
spec:
  type: {{ .Values.service.type }}
  selector:
    app: {{ .Release.Name }}-demo-app
  ports:
    - port: {{ .Values.service.port }}
      targetPort: 80
      nodePort: {{ .Values.service.nodePort }}
''')
print("Created project/demo-app-chart/ (Chart.yaml, values.yaml, templates/*)")
print("Install:   helm install demo-release project/demo-app-chart")
print("Upgrade:   helm upgrade demo-release project/demo-app-chart --set replicaCount=5")
print("Uninstall: helm uninstall demo-release")


Created project/demo-app-chart/ (Chart.yaml, values.yaml, templates/*)
Install:   helm install demo-release project/demo-app-chart
Upgrade:   helm upgrade demo-release project/demo-app-chart --set replicaCount=5
Uninstall: helm uninstall demo-release
